# TF-IDF Feature Extraction for BSR Prediction

This notebook builds TF-IDF features from product title and bullet point text, reduces them via TruncatedSVD, and computes per-category keyword profiles for explainability.

**Outputs saved to `src/web/backend/models/`:**
- `tfidf_title_vectorizer.pkl` / `tfidf_bullets_vectorizer.pkl`
- `tfidf_title_svd.pkl` / `tfidf_bullets_svd.pkl`
- `category_tfidf_profiles.pkl` - Per-category avg TF-IDF vectors of top performers
- `tfidf_feature_names.pkl` - Reverse mapping: SVD components -> top contributing terms

**New columns:** `title_tfidf_pca_0000..0049`, `bullets_tfidf_pca_0000..0049` (100 total)

In [4]:
import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy import sparse

warnings.filterwarnings('ignore')

# Paths
DATA_PATH = "../../data/products_with_image_feats.csv"
MODELS_DIR = Path("../../src/web/backend/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# TF-IDF configuration
TFIDF_MAX_FEATURES = 5000
TFIDF_MIN_DF = 5
TFIDF_MAX_DF = 0.95
TFIDF_NGRAM_RANGE = (1, 2)
SVD_N_COMPONENTS = 50  # PCA dimensions per text field

print("Loading dataset...")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} products, {len(df.columns)} columns")
print(f"\nText columns available:")
for col in ['title', 'sd_title', 'sd_feature_bullets_text', 'item_name']:
    if col in df.columns:
        non_null = df[col].notna().sum()
        print(f"  {col}: {non_null} non-null ({non_null/len(df)*100:.1f}%)")

Loading dataset...
Loaded 36879 products, 7652 columns

Text columns available:
  title: 36872 non-null (100.0%)
  sd_title: 36741 non-null (99.6%)
  sd_feature_bullets_text: 0 non-null (0.0%)


## 1. Prepare Text Fields

Build clean title and bullets text. Use `sd_title` when available, fall back to `title` column.

In [5]:
def clean_text(text):
    """Clean text for TF-IDF: lowercase, remove special chars, normalize whitespace."""
    if pd.isna(text) or not isinstance(text, str) or text.strip() == '':
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Build title text: prefer sd_title, fall back to title column
if 'sd_title' in df.columns:
    df['text_title'] = df['sd_title'].fillna(df.get('title', '')).fillna('')
else:
    df['text_title'] = df.get('title', pd.Series([''] * len(df)))

df['text_title_clean'] = df['text_title'].apply(clean_text)

# Build bullets text
if 'sd_feature_bullets_text' in df.columns:
    df['text_bullets'] = df['sd_feature_bullets_text'].fillna('')
else:
    df['text_bullets'] = ''

df['text_bullets_clean'] = df['text_bullets'].apply(clean_text)

# Stats
title_non_empty = (df['text_title_clean'].str.len() > 0).sum()
bullets_non_empty = (df['text_bullets_clean'].str.len() > 0).sum()
print(f"Title text: {title_non_empty} products with text ({title_non_empty/len(df)*100:.1f}%)")
print(f"Bullets text: {bullets_non_empty} products with text ({bullets_non_empty/len(df)*100:.1f}%)")
print(f"\nSample title: {df['text_title_clean'].iloc[0][:100]}...")
print(f"Sample bullets: {df['text_bullets_clean'].iloc[0][:100]}...")

Title text: 36872 products with text (100.0%)
Bullets text: 0 products with text (0.0%)

Sample title: bololo portable bottle warmer for travel super fast charging instant breastmilk formula water milk w...
Sample bullets: ...


## 2. Fit TF-IDF Vectorizers

Separate vectorizers for title and bullets. Using bigrams for phrase-level features.

In [6]:
# For products with empty text, fill with a placeholder so TF-IDF can handle them
# (they'll get zero vectors, which is correct behavior)
title_corpus = df['text_title_clean'].fillna('').tolist()
bullets_corpus = df['text_bullets_clean'].fillna('').tolist()

# Title TF-IDF
print("Fitting Title TF-IDF vectorizer...")
tfidf_title = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    min_df=TFIDF_MIN_DF,
    max_df=TFIDF_MAX_DF,
    ngram_range=TFIDF_NGRAM_RANGE,
    sublinear_tf=True,
    strip_accents='unicode',
    token_pattern=r'(?u)\b\w\w+\b'  # words with 2+ characters
)
title_tfidf_matrix = tfidf_title.fit_transform(title_corpus)
print(f"  Title TF-IDF shape: {title_tfidf_matrix.shape}")
print(f"  Vocabulary size: {len(tfidf_title.vocabulary_)}")
print(f"  Sample terms: {list(tfidf_title.vocabulary_.keys())[:20]}")

# Bullets TF-IDF — handle case where no bullet text exists
bullets_non_empty = sum(1 for doc in bullets_corpus if doc.strip())
print(f"\nFitting Bullets TF-IDF vectorizer...")
print(f"  Non-empty bullet documents: {bullets_non_empty}/{len(bullets_corpus)}")

if bullets_non_empty >= TFIDF_MIN_DF:
    tfidf_bullets = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        max_df=TFIDF_MAX_DF,
        ngram_range=TFIDF_NGRAM_RANGE,
        sublinear_tf=True,
        strip_accents='unicode',
        token_pattern=r'(?u)\b\w\w+\b'
    )
    bullets_tfidf_matrix = tfidf_bullets.fit_transform(bullets_corpus)
    bullets_has_vocab = True
    print(f"  Bullets TF-IDF shape: {bullets_tfidf_matrix.shape}")
    print(f"  Vocabulary size: {len(tfidf_bullets.vocabulary_)}")
    print(f"  Sample terms: {list(tfidf_bullets.vocabulary_.keys())[:20]}")
else:
    print(f"  WARNING: Not enough bullet text to fit TF-IDF (need >= {TFIDF_MIN_DF} docs).")
    print(f"  Creating zero matrix with {TFIDF_MAX_FEATURES} features as placeholder.")
    bullets_tfidf_matrix = sparse.csr_matrix((len(bullets_corpus), TFIDF_MAX_FEATURES))
    # Fit a dummy vectorizer on title corpus so it has a valid vocabulary for saving
    tfidf_bullets = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        max_df=TFIDF_MAX_DF,
        ngram_range=TFIDF_NGRAM_RANGE,
        sublinear_tf=True,
        strip_accents='unicode',
        token_pattern=r'(?u)\b\w\w+\b'
    )
    tfidf_bullets.fit(title_corpus)  # fit on titles so it has a valid state
    bullets_has_vocab = False
    print(f"  Bullets TF-IDF shape: {bullets_tfidf_matrix.shape} (all zeros)")

Fitting Title TF-IDF vectorizer...
  Title TF-IDF shape: (36879, 5000)
  Vocabulary size: 5000
  Sample terms: ['portable', 'bottle', 'warmer', 'for', 'travel', 'super', 'fast', 'charging', 'instant', 'breastmilk', 'formula', 'water', 'milk', 'with', '10', 'ounces', 'big', 'capacity', 'baby', 'flask']

Fitting Bullets TF-IDF vectorizer...
  Non-empty bullet documents: 0/36879
  Creating zero matrix with 5000 features as placeholder.
  Bullets TF-IDF shape: (36879, 5000) (all zeros)


## 3. Dimensionality Reduction with TruncatedSVD

Using TruncatedSVD (not PCA) because TF-IDF matrices are sparse. 50 components per field = 100 total new features.

In [7]:
# Title SVD
print("Fitting Title TruncatedSVD...")
svd_title = TruncatedSVD(n_components=SVD_N_COMPONENTS, random_state=42)
title_svd_matrix = svd_title.fit_transform(title_tfidf_matrix)
print(f"  Title SVD shape: {title_svd_matrix.shape}")
print(f"  Explained variance: {svd_title.explained_variance_ratio_.sum():.3f}")

# Bullets SVD
print("\nFitting Bullets TruncatedSVD...")
svd_bullets = TruncatedSVD(n_components=SVD_N_COMPONENTS, random_state=42)
bullets_svd_matrix = svd_bullets.fit_transform(bullets_tfidf_matrix)
print(f"  Bullets SVD shape: {bullets_svd_matrix.shape}")
print(f"  Explained variance: {svd_bullets.explained_variance_ratio_.sum():.3f}")

# Append SVD columns to dataframe
print("\nAppending SVD features to dataframe...")
for i in range(SVD_N_COMPONENTS):
    df[f'title_tfidf_pca_{i:04d}'] = title_svd_matrix[:, i]
    df[f'bullets_tfidf_pca_{i:04d}'] = bullets_svd_matrix[:, i]

tfidf_cols = [c for c in df.columns if c.startswith('title_tfidf_pca_') or c.startswith('bullets_tfidf_pca_')]
print(f"  Added {len(tfidf_cols)} TF-IDF SVD columns")
print(f"  Total dataframe shape: {df.shape}")

Fitting Title TruncatedSVD...
  Title SVD shape: (36879, 50)
  Explained variance: 0.188

Fitting Bullets TruncatedSVD...
  Bullets SVD shape: (36879, 50)
  Explained variance: nan

Appending SVD features to dataframe...
  Added 100 TF-IDF SVD columns
  Total dataframe shape: (36879, 7756)


## 4. Build Reverse Mapping (SVD Components -> Top Terms)

For explainability: maps each SVD component to the TF-IDF terms that contribute most.

In [8]:
def build_svd_term_mapping(svd_model, vectorizer, top_n=15):
    """
    Build reverse mapping from SVD components to top contributing TF-IDF terms.
    
    For each SVD component, returns the top_n terms with highest absolute loadings.
    This lets us explain what each latent dimension represents.
    """
    feature_names = vectorizer.get_feature_names_out()
    component_terms = {}
    
    for comp_idx in range(svd_model.n_components):
        loadings = svd_model.components_[comp_idx]
        # Get top terms by absolute magnitude
        top_indices = np.argsort(np.abs(loadings))[-top_n:][::-1]
        terms_with_weights = [
            {'term': feature_names[idx], 'weight': float(loadings[idx])}
            for idx in top_indices
        ]
        component_terms[comp_idx] = terms_with_weights
    
    return component_terms

# Build mappings
title_component_terms = build_svd_term_mapping(svd_title, tfidf_title)

if bullets_has_vocab:
    bullets_component_terms = build_svd_term_mapping(svd_bullets, tfidf_bullets)
else:
    bullets_component_terms = {}
    print("Bullets SVD component mapping skipped (no bullet text available)")

# Show top 3 components for title
print("Title SVD Component -> Top Terms:")
for comp_idx in range(3):
    terms = [t['term'] for t in title_component_terms[comp_idx][:8]]
    print(f"  Component {comp_idx}: {', '.join(terms)}")

if bullets_component_terms:
    print("\nBullets SVD Component -> Top Terms:")
    for comp_idx in range(3):
        terms = [t['term'] for t in bullets_component_terms[comp_idx][:8]]
        print(f"  Component {comp_idx}: {', '.join(terms)}")
else:
    print("\nBullets: no component terms (empty corpus)")

Bullets SVD component mapping skipped (no bullet text available)
Title SVD Component -> Top Terms:
  Component 0: for, with, and, dog, pack, large, black, baby
  Component 1: women, men, for women, for men, high, pants, yoga, leggings
  Component 2: dog, dogs, pet, medium, desk, office, small, large

Bullets: no component terms (empty corpus)


## 5. Build Per-Category TF-IDF Profiles

Compute average TF-IDF vectors for **top performing products** (BSR rank in top 25%) per category. These profiles are used at inference time to compare a user's product text against successful competitors and generate keyword suggestions.

In [9]:
# Build category TF-IDF profiles for keyword comparison
# For each category, compute mean TF-IDF vector of top-performing products
category_tfidf_profiles = {}

# Categories from the modeling pipeline
CATEGORIES = [
    "Home & Kitchen", "Health & Household", "Office Products", "Baby",
    "Clothing, Shoes & Jewelry", "Kitchen & Dining", "Electronics",
    "Cell Phones & Accessories", "Tools & Home Improvement", "Video Games",
    "Pet Supplies", "Sports & Outdoors", "Industrial & Scientific",
    "Musical Instruments",
]

# Need category column - check which one to use
cat_col = 'main_bsr_group' if 'main_bsr_group' in df.columns else 'category'
print(f"Using category column: {cat_col}")

for category in CATEGORIES:
    cat_mask = df[cat_col] == category
    cat_df = df[cat_mask & df['main_bsr_rank'].notna()].copy()
    
    if len(cat_df) < 10:
        print(f"  {category}: only {len(cat_df)} products with BSR, skipping")
        continue
    
    # Top 25% = lowest BSR rank (lower rank = better selling)
    rank_threshold = cat_df['main_bsr_rank'].quantile(0.25)
    top_mask = cat_df['main_bsr_rank'] <= rank_threshold
    top_indices = cat_df[top_mask].index
    
    # Get indices in the original df for these top products
    # Map to position indices for the TF-IDF matrices
    pos_indices = [df.index.get_loc(idx) for idx in top_indices if idx in df.index]
    
    if len(pos_indices) < 5:
        print(f"  {category}: only {len(pos_indices)} top performers, skipping")
        continue
    
    # Compute mean TF-IDF for top performers (title)
    title_avg = title_tfidf_matrix[pos_indices].mean(axis=0)
    title_avg = np.asarray(title_avg).flatten()
    
    # Compute mean TF-IDF for top performers (bullets)
    bullets_avg = bullets_tfidf_matrix[pos_indices].mean(axis=0)
    bullets_avg = np.asarray(bullets_avg).flatten()
    
    # Also compute keyword prevalence: fraction of top products using each term
    title_binary = (title_tfidf_matrix[pos_indices] > 0).astype(float)
    title_prevalence = np.asarray(title_binary.mean(axis=0)).flatten()
    
    bullets_binary = (bullets_tfidf_matrix[pos_indices] > 0).astype(float)
    bullets_prevalence = np.asarray(bullets_binary.mean(axis=0)).flatten()
    
    category_tfidf_profiles[category] = {
        'title_avg_tfidf': title_avg,
        'title_prevalence': title_prevalence,
        'bullets_avg_tfidf': bullets_avg,
        'bullets_prevalence': bullets_prevalence,
        'n_top_products': len(pos_indices),
        'rank_threshold': float(rank_threshold),
    }
    
    # Show top keywords for this category
    title_names = tfidf_title.get_feature_names_out()
    top_title_idx = np.argsort(title_prevalence)[-5:][::-1]
    top_keywords = [(title_names[i], f"{title_prevalence[i]*100:.0f}%") for i in top_title_idx]
    print(f"  {category}: {len(pos_indices)} top products, top title keywords: {top_keywords}")

print(f"\nBuilt profiles for {len(category_tfidf_profiles)} categories")

Using category column: main_bsr_group
  Home & Kitchen: 943 top products, top title keywords: [('for', '69%'), ('with', '48%'), ('and', '37%'), ('set', '30%'), ('storage', '24%')]
  Health & Household: 918 top products, top title keywords: [('for', '30%'), ('count', '30%'), ('and', '25%'), ('pack', '25%'), ('oz', '24%')]
  Office Products: 855 top products, top title keywords: [('for', '61%'), ('office', '39%'), ('with', '38%'), ('pack', '26%'), ('paper', '25%')]
  Baby: 726 top products, top title keywords: [('baby', '78%'), ('for', '57%'), ('with', '41%'), ('pack', '33%'), ('and', '33%')]
  Clothing, Shoes & Jewelry: 535 top products, top title keywords: [('men', '39%'), ('women', '39%'), ('for', '31%'), ('socks', '18%'), ('with', '16%')]
  Kitchen & Dining: 433 top products, top title keywords: [('with', '55%'), ('for', '44%'), ('steel', '42%'), ('stainless', '40%'), ('stainless steel', '40%')]
  Electronics: 411 top products, top title keywords: [('with', '53%'), ('for', '53%'), ('

## 6. Save All Artifacts

In [10]:
# Save TF-IDF vectorizers
with open(MODELS_DIR / 'tfidf_title_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_title, f)
print(f"Saved tfidf_title_vectorizer.pkl")

with open(MODELS_DIR / 'tfidf_bullets_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_bullets, f)
print(f"Saved tfidf_bullets_vectorizer.pkl" + (" (fitted on titles as placeholder)" if not bullets_has_vocab else ""))

# Save SVD transformers
with open(MODELS_DIR / 'tfidf_title_svd.pkl', 'wb') as f:
    pickle.dump(svd_title, f)
print(f"Saved tfidf_title_svd.pkl")

with open(MODELS_DIR / 'tfidf_bullets_svd.pkl', 'wb') as f:
    pickle.dump(svd_bullets, f)
print(f"Saved tfidf_bullets_svd.pkl")

# Save category profiles
with open(MODELS_DIR / 'category_tfidf_profiles.pkl', 'wb') as f:
    pickle.dump(category_tfidf_profiles, f)
print(f"Saved category_tfidf_profiles.pkl ({len(category_tfidf_profiles)} categories)")

# Save reverse mapping
tfidf_feature_names = {
    'title_component_terms': title_component_terms,
    'bullets_component_terms': bullets_component_terms,
    'title_feature_names': list(tfidf_title.get_feature_names_out()),
    'bullets_feature_names': list(tfidf_bullets.get_feature_names_out()) if bullets_has_vocab else [],
    'bullets_has_vocab': bullets_has_vocab,
}
with open(MODELS_DIR / 'tfidf_feature_names.pkl', 'wb') as f:
    pickle.dump(tfidf_feature_names, f)
print(f"Saved tfidf_feature_names.pkl")

# Save the enriched dataframe for the next notebook
OUTPUT_CSV = "../../data/products_with_image_feats.csv"
print(f"\nSaving enriched dataframe to {OUTPUT_CSV}...")
print(f"  Shape: {df.shape}")
df.to_csv(OUTPUT_CSV, index=False)
print("Done! TF-IDF features added to dataset.")

Saved tfidf_title_vectorizer.pkl
Saved tfidf_bullets_vectorizer.pkl (fitted on titles as placeholder)
Saved tfidf_title_svd.pkl
Saved tfidf_bullets_svd.pkl
Saved category_tfidf_profiles.pkl (14 categories)
Saved tfidf_feature_names.pkl

Saving enriched dataframe to ../../data/products_with_image_feats.csv...
  Shape: (36879, 7756)
Done! TF-IDF features added to dataset.
